# 🧠 FaceKey Studio Gen2 — Google Colab T4 GPU Model Trainer
### Accelerated Speech-to-Facial Animation Training with Background Execution & Zero Data Loss

## 🌟 HOW TO RUN IN BACKGROUND WHEN BROWSER CLOSES:
1. **Option A (Background Detached Runner — Recommended)**:
   Run Step 4B (`nohup` background launch). The training process runs detached inside the Colab VM. You can inspect live progress via the Log Viewer in Step 5.
2. **Option B (Direct Google Drive State Mounting)**:
   All checkpoints (`checkpoint_latest.pt`, `checkpoint_best.pt`, `loss_curve.png`) are saved **directly to your mounted 5 TB Google Drive** atomically every epoch.
   If Colab ever disconnects when you close the tab, **0 bytes of training are lost**.
3. **Option C (1-Click Auto-Resume)**:
   Whenever you reopen this notebook, click **Run All**. It automatically detects `checkpoint_latest.pt` on Google Drive and resumes from the exact next epoch with the AdamW optimizer state intact!
4. **Option D (Colab Pro Native Background Execution)**:
   If you have Colab Pro, click **Runtime** ➔ **Run in background** to allow headless cloud execution.

**🛡️ GEN2 ARCHITECTURAL ENHANCEMENTS:**
- 64-band Log-Mel spectral filterbanks.
- 5-state Emotional Expression Conditioning (`Neutral`, `Happy`, `Angry`, `Scared`, `Sad`).
- 3D Head Orientation Kinematics (`Pitch`, `Yaw`, `Roll`) with velocity loss.
- Lingual Tongue Invisibility Guard (`tongueOut`).

## 💓 STEP 0: BROWSER ANTI-SLEEP HEARTBEAT
> Prevents background tab throttling and idle disconnect.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive(){
    console.log('[FaceKey Heartbeat] Colab session kept alive at ' + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('[✓] Browser Keep-Alive Heartbeat Activated!')


## ⚡ STEP 1: VERIFY NVIDIA GPU ACCELERATION

In [ ]:
!nvidia-smi

import torch
print(f'[+] PyTorch CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[+] Active GPU: {torch.cuda.get_device_name(0)}')
else:
    print('[!] WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.')


## 📁 STEP 2: MOUNT 5 TB GOOGLE DRIVE
Connects your Google Drive containing dataset chunks and checkpoint directories.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'[✓] Google Drive mounted successfully at: {DRIVE_DIR}')


## ⚡ STEP 3: CLONE & UPDATE FACEKEY GEN2 REPOSITORY (SELF-HEALING)

In [ ]:
import os, sys
%cd /content

REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/content/FaceKeyAnimation'

if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating existing repository to latest origin/main...')
    !cd {APP_DIR} && git fetch --all --prune && git checkout main && git reset --hard origin/main && git clean -fd
else:
    print('[*] Performing fresh clone of FaceKeyAnimation (Gen2 branch)...')
    !rm -rf {APP_DIR}
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}
!pip install -q -U torch torchvision torchaudio huggingface_hub numpy scipy
print('[+] Training environment ready with latest Gen2 codebase!')


## 🚀 STEP 4A: LAUNCH TRAINING (INTERACTIVE LIVE MODE)
> Run this cell if you want to see live epoch-by-epoch loss updates in your notebook.

In [ ]:
# Run interactively: auto-saves checkpoints to Google Drive & Hugging Face Hub every epoch
!python scripts/train_speech_to_animation.py \
    --drive-dir "/content/drive/MyDrive/FaceKeyDataset" \
    --hf-repo "VijayTheOne/facekey-dataset-chunks" \
    --epochs 50 \
    --batch-size 64 \
    --lr 0.0001 \
    --seq-len 64 \
    --stride 16 \
    --accum-steps 2


## 🌙 STEP 4B: LAUNCH TRAINING IN BACKGROUND (BROWSER-CLOSE SAFE)
> Run this cell to start training as a **detached background process (`nohup`)**.
> You can safely close your browser tab — training continues in the background on the Colab VM!

In [ ]:
# Launch training detached in background
import subprocess

LOG_FILE = '/content/drive/MyDrive/FaceKeyDataset/checkpoints/colab_training.log'
cmd = f'''
nohup python scripts/train_speech_to_animation.py \
    --drive-dir "/content/drive/MyDrive/FaceKeyDataset" \
    --hf-repo "VijayTheOne/facekey-dataset-chunks" \
    --epochs 50 \
    --batch-size 64 \
    --lr 0.0001 \
    --seq-len 64 \
    --stride 16 \
    --accum-steps 2 > {LOG_FILE} 2>&1 &
'''
subprocess.Popen(cmd, shell=True, executable='/bin/bash')
print(f'[✓] FaceKey Gen2 Trainer launched in background!')
print(f'[*] Continuous Log Location: {LOG_FILE}')
print('[*] You can now safely close this browser tab!')


## 📋 STEP 5: MONITOR BACKGROUND TRAINING PROGRESS & LIVE LOGS

In [ ]:
# View live background training telemetry
import os, time
LOG_FILE = '/content/drive/MyDrive/FaceKeyDataset/checkpoints/colab_training.log'
if os.path.exists(LOG_FILE):
    !tail -n 25 {LOG_FILE}
else:
    print('[*] Log file not found yet. Training may be starting...')


## 📈 STEP 6: VIEW LOSS CURVE PLOT

In [ ]:
import os
from IPython.display import Image, display

plot_path = '/content/drive/MyDrive/FaceKeyDataset/checkpoints/loss_curve.png'
if os.path.exists(plot_path):
    display(Image(filename=plot_path))
else:
    print('[*] Loss curve will appear after Epoch 1 completes.')
